# 02 Cleaning and entity resolution

**Owner:** Person A.

**Inputs:** `data/raw/` and `outputs/column_roles.json`.

**Outputs:** `data/processed/analysis_frame.parquet` (or csv if parquet refuses a column), `outputs/alias_map.csv`, `outputs/tables/unmatched_keys.csv`, `outputs/tables/fuzzy_candidates.csv`.

**Stop rule:** `APPLY_FUZZY` stays false. Fuzzy pairs are a list for a person to read. If a join multiplies the row count by more than 1.5, the analysis frame is not written.

Push this frame as soon as it exists. Person B and Person C pull before they continue.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.paths import (
    RAW,
    PROCESSED,
    OUTPUTS,
    TABLES,
    ANALYSIS_PATH,
    FEATURES_PATH,
    MODEL_MATRIX_PATH,
    ROLES_PATH,
    CONTRACT_PATH,
    GATE_PATH,
    FEATURE_LOG_PATH,
    ALIAS_PATH,
    ANSWERS_PATH,
    COMPARISON_PATH,
    BUNDLE_PATH,
    ensure_dirs,
)

ensure_dirs()
print("root", ROOT)


In [ ]:
import pandas as pd

from src import io as sio
from src.entity import alias_table, fuzzy_candidates, join_on_key, key_overlap, unmatched_keys, with_keys
from src.roles import columns_with_role, read_roles, validate_roles

APPLY_FUZZY = False

roles = read_roles()
files = sio.list_raw_files(RAW)
if not files:
    print("STOP: no files in data/raw.")
else:
    frames = [(path, sio.load_table(path)) for path in files]
    for path, frame in frames:
        report = validate_roles(roles, frame.columns)
        pending = {name: values for name, values in report.items() if values}
        print(path.name, pending or "roles cover this file")
    id_columns = columns_with_role(roles, "id")
    drop_columns = columns_with_role(roles, "drop")

    def without_drops(frame):
        return frame.drop(columns=[column for column in drop_columns if column in frame.columns])

    if len(frames) == 1:
        path, frame = frames[0]
        id_column = next((column for column in id_columns if column in frame.columns), None)
        cleaned = without_drops(frame)
        if id_column:
            cleaned = with_keys(cleaned, id_column)
            alias = alias_table(frame[id_column], path.stem)
            alias.to_csv(ALIAS_PATH, index=False)
            print("alias rows", len(alias), "distinct keys", alias["normalized_id"].nunique())
        written = sio.save_frame(cleaned.reset_index(drop=True), ANALYSIS_PATH)
        print("wrote", written, cleaned.shape)
    else:
        (left_path, left), (right_path, right) = frames[0], frames[1]
        left_id = next((column for column in id_columns if column in left.columns), None)
        right_id = next((column for column in id_columns if column in right.columns), None)
        if left_id is None or right_id is None:
            print("STOP: tag an id column on both tables in outputs/column_roles.json.")
        else:
            print("exact normalized overlap", key_overlap(left[left_id], right[right_id]))
            alias = pd.concat(
                [alias_table(left[left_id], left_path.stem), alias_table(right[right_id], right_path.stem)],
                ignore_index=True,
            )
            alias.to_csv(ALIAS_PATH, index=False)
            unmatched_keys(left[left_id], right[right_id]).to_csv(TABLES / "unmatched_keys.csv", index=False)
            pairs = fuzzy_candidates(left[left_id], right[right_id])
            pairs.to_csv(TABLES / "fuzzy_candidates.csv", index=False)
            print("fuzzy candidates listed, not merged:", len(pairs))
            if APPLY_FUZZY:
                raise RuntimeError("Leave APPLY_FUZZY false until someone has accepted the pairs.")
            merged = join_on_key(without_drops(left), without_drops(right), left_id, right_id, how="left")
            print("rows", len(left), "->", len(merged))
            if len(merged) > 1.5 * max(len(left), 1):
                print("STOP: the join multiplied rows. Fix duplicate keys before writing the analysis frame.")
            else:
                written = sio.save_frame(merged.reset_index(drop=True), ANALYSIS_PATH)
                print("wrote", written, merged.shape)
        if len(frames) > 2:
            print("More than two raw files. This notebook joins the first two only. Say so on the slide if the others are unused.")
